# 🎭 FaceLab (DeepfakeProject) — Google Colab com GPU

**Repositório original:** https://github.com/CelinDev/deepfakeproject (código em `FaceLab/`)

Este notebook executa o **pipeline real de vídeo offline** do FaceLab (`src/pipeline/video_pipeline.py::VideoPipeline`),
o mesmo usado por `scripts/run_swap.py` e pela aba "Vídeo Offline" da interface desktop.

### O que o pipeline faz (conforme o código-fonte)
| Etapa | Módulo | Modelo | Execução |
|---|---|---|---|
| Detecção + 5 landmarks + embedding | `src/face/face_detector.py` | InsightFace `buffalo_l` (SCRFD `det_10g` + ArcFace `w600k_r50`) | ONNX Runtime (CUDA EP) |
| Troca facial | `src/face/face_swapper.py` | `inswapper_128.onnx` | ONNX Runtime (CUDA EP) |
| Segmentação cabelo/orelhas (opcional) | `src/face/face_parser.py` | `bisenet_resnet_34.onnx` | ONNX Runtime (CUDA EP) |
| Restauração facial (opcional) | `src/face/face_enhancer.py` | `gfpgan_1.4.onnx` | ONNX Runtime (CUDA EP) |
| Alinhamento, composição, suavização temporal, rastreamento | `face_aligner`, `face_compositor`, `temporal_smoother`, `face_tracker` | — | **CPU** (OpenCV/NumPy) |
| Leitura/gravação de vídeo + áudio + metadado de IA | `src/video/frame_source.py`, `video_writer.py` | — | **CPU** (FFmpeg) |

> O PyTorch é usado **apenas** para detectar a GPU (`DeviceManager`) e para carregar as DLLs CUDA antes do ONNX Runtime.
> **Nenhuma inferência é feita em PyTorch**, por isso `torch.inference_mode()`/AMP não se aplicam (ver Célula 9).

### Requisitos
- **GPU NVIDIA**: menu **Ambiente de execução → Alterar tipo de ambiente de execução → Acelerador de hardware: GPU** (T4 basta).
- **Disco**: ~2 GB para os modelos (+ espaço para vídeos).
- **VRAM**: ~2–4 GB com todos os modelos carregados (estimativa, não medida em todas as GPUs).

### Limitações conhecidas
- A **webcam ao vivo** (`LivePipeline`) e a **interface PySide6** não funcionam no Colab (sem câmera local / sem display). Apenas o fluxo de vídeo é suportado.
- O processamento é **quadro a quadro** (o código original não suporta inferência em lote).
- Pesos `buffalo_l` e `inswapper_128`: **uso não comercial** (ver `FaceLab/models/README.md`).

### ⚠️ Uso ético
Use apenas vídeos e rostos **com autorização** das pessoas retratadas. O vídeo final recebe o metadado
*"Conteudo sintetico: video alterado por IA"* gravado pelo próprio projeto (`AI_DISCLOSURE`) — não o remova.

## 1️⃣ Diagnóstico do ambiente

In [ ]:
# Diagnóstico somente-leitura: não instala nada.
import importlib.metadata as md_, platform, shutil, subprocess, sys

def versao(pkg):
    try:
        return md_.version(pkg)
    except md_.PackageNotFoundError:
        return "não instalado"

print(f"Python      : {sys.version.split()[0]} ({platform.platform()})")
try:
    import torch
    print(f"PyTorch     : {torch.__version__} | CUDA do build: {torch.version.cuda}")
    print(f"CUDA ativa  : {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        cap = torch.cuda.get_device_capability(0)
        print(f"GPU         : {torch.cuda.get_device_name(0)} | {p.total_memory/1024**3:.1f} GB | sm_{cap[0]}{cap[1]}")
    else:
        print("⚠️  Sem GPU. Ative-a em Ambiente de execução → Alterar tipo → GPU e execute de novo.")
except ImportError:
    print("PyTorch     : não instalado")

for pkg in ["numpy", "opencv-python", "opencv-python-headless", "onnxruntime", "onnxruntime-gpu",
            "insightface", "onnx", "scikit-image", "scipy", "albumentations"]:
    print(f"{pkg:22s}: {versao(pkg)}")

ff = shutil.which("ffmpeg")
print(f"ffmpeg      : {ff or 'NÃO ENCONTRADO'}")
if ff:
    print("              " + subprocess.run([ff, "-version"], capture_output=True, text=True).stdout.splitlines()[0])
du = shutil.disk_usage("/content")
print(f"Disco livre : {du.free/1024**3:.1f} GB")
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

## 2️⃣ Configuração centralizada
Todos os parâmetros ficam **somente aqui**. Os nomes correspondem aos argumentos reais de `VideoPipeline`.

In [ ]:
# @title Parâmetros { display-mode: "form" }
REPO_URL   = "https://github.com/CelinDev/deepfakeproject.git"  # @param {type:"string"}
REPO_REF   = "main"            # @param {type:"string"}
WORKDIR    = "/content/deepfakeproject"  # @param {type:"string"}

# Persistência opcional no Google Drive (modelos e resultados)
USE_DRIVE        = False       # @param {type:"boolean"}
DRIVE_BASE       = "/content/drive/MyDrive/FaceLab"  # @param {type:"string"}

# --- Parâmetros do VideoPipeline (mesmos do código original) ---
DEVICE           = "auto"      # @param ["auto", "gpu", "cpu"]
ENABLE_ENHANCER  = False       # @param {type:"boolean"}
ENABLE_HEAD_SWAP = True        # @param {type:"boolean"}
BLEND_INTENSITY  = 1.0         # @param {type:"slider", min:0.0, max:1.0, step:0.05}
COLOR_CORRECT    = True        # @param {type:"boolean"}
FEATHER_AMOUNT   = 15          # @param {type:"slider", min:3, max:45, step:2}

# Rosto de origem: "upload" (sua foto) ou um dos presets incluídos em FaceLab/assets
SOURCE_MODE      = "upload"    # @param ["upload", "zumbi", "vampiro", "ciborgue"]

# --- Pré-processamento opcional (desligado = vídeo original intacto) ---
MAX_SECONDS      = 0           # @param {type:"integer"}
MAX_HEIGHT       = 0           # @param {type:"integer"}

# Confirmação obrigatória de consentimento
CONSENT = False  # @param {type:"boolean"}

from pathlib import Path
FACELAB   = Path(WORKDIR) / "FaceLab"
INPUT_DIR = Path("/content/inputs")
INPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Configuração carregada. FaceLab em:", FACELAB)

### (Opcional) Montar o Google Drive
Guarda os modelos (~2 GB) em `DRIVE_BASE/models` para não baixá-los a cada sessão, e permite copiar resultados.

In [ ]:
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(DRIVE_BASE, "models").mkdir(parents=True, exist_ok=True)
    Path(DRIVE_BASE, "output").mkdir(parents=True, exist_ok=True)
    print("Drive montado:", DRIVE_BASE)
else:
    print("USE_DRIVE=False — tudo fica em /content (perdido ao encerrar a sessão).")

## 3️⃣ Clonar o repositório

In [ ]:
import os, subprocess, sys

def run(cmd, **kw):
    # Executa um comando e PARA o notebook se falhar (sem esconder erros).
    print("$", " ".join(map(str, cmd)))
    r = subprocess.run(cmd, text=True, capture_output=True, **kw)
    print(r.stdout[-3000:], r.stderr[-3000:], sep="\n")
    if r.returncode != 0:
        raise RuntimeError(f"Comando falhou (código {r.returncode}): {cmd}")
    return r

if not (Path(WORKDIR) / ".git").is_dir():
    run(["git", "clone", "--branch", REPO_REF, REPO_URL, WORKDIR])
else:
    print("Repositório já existe; atualizando sem descartar alterações locais.")
    run(["git", "-C", WORKDIR, "pull", "--ff-only"])

run(["git", "-C", WORKDIR, "log", "-1", "--oneline"])
assert (FACELAB / "src" / "pipeline" / "video_pipeline.py").is_file(), "Estrutura FaceLab/ não encontrada"

os.chdir(FACELAB)                      # config/settings.py usa caminhos relativos à raiz FaceLab
if str(FACELAB) not in sys.path:
    sys.path.insert(0, str(FACELAB))

# Modelos persistentes no Drive: substitui FaceLab/models por link simbólico (apenas se ainda não existir conteúdo).
if USE_DRIVE:
    local_models = FACELAB / "models"
    drive_models = Path(DRIVE_BASE, "models")
    if not local_models.is_symlink():
        # move o README versionado para o Drive em vez de apagá-lo
        for f in local_models.iterdir():
            dest = drive_models / f.name
            if not dest.exists():
                f.rename(dest)
        local_models.rmdir()
        local_models.symlink_to(drive_models, target_is_directory=True)
    print("models/ →", os.readlink(local_models))
print("Diretório atual:", Path.cwd())

## 4️⃣ Instalação inteligente das dependências

Análise do `FaceLab/requirements.txt` frente ao Colab:

| Pacote original | Decisão no Colab | Motivo |
|---|---|---|
| `torch`/`torchvision` (cu126 via índice próprio) | **Não reinstalar** | O Colab já traz PyTorch com CUDA compatível com a GPU da sessão. |
| `numpy`, `opencv-python`, `scipy`, `scikit-image`, `requests`, `tqdm` | Usar os do Colab; instalar só se faltar | Evita quebrar o ambiente base. |
| `onnxruntime-gpu==1.23.2` | **Manter a versão fixada** | ≥1.24 usa CUDA 13; o Colab usa CUDA 12 (verificado na célula 2). |
| `onnxruntime` (CPU) | **Remover se presente** | Sobrescreve o `onnxruntime-gpu` (aviso no próprio requirements). |
| `insightface==2.1` | Instalar com `--no-deps` | Instrução do projeto: evita puxar `onnxruntime` CPU. |
| `onnx` | Instalar se faltar | Usado por `face_swapper.py` para ler a matriz `emap`. |
| `PySide6`, `pytest` | **Não instalar** | GUI e testes não são usados no Colab. |

In [ ]:
import importlib.metadata as md_

def instalado(pkg):
    try:
        return md_.version(pkg)
    except md_.PackageNotFoundError:
        return None

def pip(*args):
    run([sys.executable, "-m", "pip", "install", "-q", *args])

ORT_VERSION = "1.23.2"   # fixado no requirements.txt original

# 1) onnxruntime CPU conflita com o GPU
if instalado("onnxruntime"):
    run([sys.executable, "-m", "pip", "uninstall", "-y", "onnxruntime"])

# 2) onnxruntime-gpu na versão do projeto
if instalado("onnxruntime-gpu") != ORT_VERSION:
    pip(f"onnxruntime-gpu=={ORT_VERSION}")

# 3) dependências leves: somente as que faltam (não atualiza as existentes)
faltando = [p for p in ["onnx", "scikit-image", "scipy", "requests", "tqdm", "albumentations"] if not instalado(p)]
if not instalado("opencv-python") and not instalado("opencv-python-headless"):
    faltando.append("opencv-python-headless")
if faltando:
    pip(*faltando)

# 4) insightface sem dependências (para não trazer onnxruntime CPU)
if instalado("insightface") is None:
    pip("insightface==2.1", "--no-deps")

# Garantia final: nada reinstalou o onnxruntime CPU
assert instalado("onnxruntime") is None, "onnxruntime (CPU) voltou a ser instalado; remova-o."
print("\nVersões finais:")
for p in ["torch", "numpy", "onnxruntime-gpu", "insightface", "onnx", "opencv-python", "opencv-python-headless"]:
    print(f"  {p:24s} {instalado(p)}")
print("\nℹ️ NumPy/PyTorch não foram alterados → reinício da sessão NÃO é necessário.")

In [ ]:
# Teste de importação: falha aqui = problema de instalação, antes de baixar 2 GB de modelos.
import importlib, traceback
erros = []
for mod in ["torch", "onnxruntime", "onnx", "cv2", "numpy", "skimage", "insightface", "insightface.app",
            "config.settings", "src.inference.model_loader", "src.pipeline.video_pipeline"]:
    try:
        importlib.import_module(mod)
        print(f"✅ {mod}")
    except Exception:
        erros.append(mod)
        print(f"❌ {mod}\n{traceback.format_exc()}")
if erros:
    raise ImportError(f"Falha ao importar: {erros}. Veja a seção de diagnóstico no final.")

## 5️⃣ Configuração prioritária da GPU

O projeto já seleciona o dispositivo via `DeviceManager` e configura o ONNX Runtime em
`src/inference/model_loader.py::onnx_providers` (`CUDAExecutionProvider` → `CPUExecutionProvider`).
**Nenhuma alteração no código original foi necessária.** Aqui apenas verificamos que a CUDA EP realmente carrega.

In [ ]:
import torch                    # precisa vir antes do onnxruntime (docstring de model_loader.py)
import onnxruntime as ort
from src.inference.device_manager import DeviceManager, DevicePreference

# ORT >= 1.21: pré-carrega CUDA/cuDNN das wheels nvidia-* instaladas com o PyTorch
if hasattr(ort, "preload_dlls"):
    ort.preload_dlls()

dm = DeviceManager(DevicePreference(DEVICE))
print("Dispositivo do projeto :", dm.info.name, "| CUDA:", dm.info.is_cuda)
print("Providers disponíveis  :", ort.get_available_providers())

if dm.info.is_cuda and "CUDAExecutionProvider" not in ort.get_available_providers():
    raise RuntimeError("GPU detectada pelo PyTorch, mas o onnxruntime-gpu não expõe CUDAExecutionProvider. "
                       "Veja 'CUDA EP indisponível' no diagnóstico.")
if not dm.info.is_cuda:
    print("⚠️ Executando em CPU: o processamento será MUITO mais lento.")

# TF32: o ONNX Runtime CUDA EP usa a opção 'use_tf32' (padrão ligado) em GPUs Ampere+; T4 (Turing) não tem TF32.
# FP16: os modelos .onnx são FP32; convertê-los alteraria os resultados, por isso NÃO é aplicado.

## 6️⃣ Download dos modelos (fontes do próprio projeto)

Usa o script original `scripts/download_models.py`, que já pula arquivos existentes. URLs **do código-fonte**:

| Arquivo | Fonte (do repositório) | Tamanho aprox. |
|---|---|---|
| `inswapper_128.onnx` | huggingface.co/facefusion/models-3.0.0 | ~554 MB |
| `gfpgan_1.4.onnx` | huggingface.co/facefusion/models-3.0.0 | ~340 MB |
| `buffalo_l.zip` | github.com/deepinsight/insightface releases v0.7 | ~280 MB (+ extraído) |
| `bisenet_resnet_34.onnx` | huggingface.co/facefusion/models-3.0.0 (de `face_parser.py`) | ~94 MB |

O repositório **não fornece checksums**; a integridade é validada abrindo cada modelo no ONNX Runtime (próxima célula).

In [ ]:
run([sys.executable, "scripts/download_models.py"])

# BiSeNet: o FaceParser baixa sozinho na 1ª execução; antecipamos aqui com a MESMA URL do código.
from config.settings import BISENET_MODEL_PATH, MODELS_DIR
if ENABLE_HEAD_SWAP and not BISENET_MODEL_PATH.is_file():
    import urllib.request
    urllib.request.urlretrieve(
        "https://huggingface.co/facefusion/models-3.0.0/resolve/main/bisenet_resnet_34.onnx",
        str(BISENET_MODEL_PATH))

for f in sorted(MODELS_DIR.rglob("*.onnx")):
    print(f"{f.relative_to(MODELS_DIR)!s:45s} {f.stat().st_size/1024**2:8.1f} MB")

In [ ]:
# Validação de integridade: cada modelo precisa abrir no ONNX Runtime com os providers do projeto.
from src.inference.model_loader import onnx_providers
from config.settings import MODELS_DIR, BISENET_MODEL_PATH

obrig = [MODELS_DIR / "inswapper_128.onnx",
         MODELS_DIR / "models" / "buffalo_l" / "det_10g.onnx",
         MODELS_DIR / "models" / "buffalo_l" / "w600k_r50.onnx"]
if ENABLE_ENHANCER:  obrig.append(MODELS_DIR / "gfpgan_1.4.onnx")
if ENABLE_HEAD_SWAP: obrig.append(BISENET_MODEL_PATH)

for p in obrig:
    if not p.is_file():
        raise FileNotFoundError(f"Modelo ausente: {p}")
    s = ort.InferenceSession(str(p), providers=onnx_providers(dm))
    print(f"✅ {p.name:28s} providers={s.get_providers()}")
    del s

## 7️⃣ Arquivos de entrada
Formatos aceitos pelo projeto (`config/settings.py`): vídeo `.mp4 .mov .avi .mkv .webm`; imagem `.jpg .jpeg .png .bmp .webp`.
Você pode fazer upload ou definir caminhos (por exemplo no Drive) nas variáveis abaixo.

In [ ]:
# Deixe vazio para fazer upload; ou informe um caminho existente (ex.: /content/drive/MyDrive/video.mp4)
VIDEO_PATH  = ""  # @param {type:"string"}
SOURCE_PATH = ""  # @param {type:"string"}

from config.settings import (SUPPORTED_VIDEO_EXTENSIONS, SUPPORTED_IMAGE_EXTENSIONS,
                             ZOMBIE_REFERENCE_IMAGE, VAMPIRE_REFERENCE_IMAGE, CYBORG_REFERENCE_IMAGE)
from src.utils.validators import validate_video_path, validate_image_path

def upload(descricao, exts):
    from google.colab import files
    print(f"Selecione {descricao} ({', '.join(sorted(exts))})")
    up = files.upload()
    if not up:
        raise ValueError("Nenhum arquivo enviado.")
    nome, dados = next(iter(up.items()))
    destino = INPUT_DIR / Path(nome).name
    destino.write_bytes(dados)
    return destino

if not CONSENT:
    raise PermissionError("Marque CONSENT=True na configuração confirmando que você tem autorização das pessoas retratadas.")

video = Path(VIDEO_PATH) if VIDEO_PATH else upload("o vídeo de entrada", SUPPORTED_VIDEO_EXTENSIONS)
video = validate_video_path(video)          # validador original do projeto

# Mapeamento de presets idêntico ao da interface (main_window.py).
# Zumbi: o pipeline usa ZOMBIE_REFERENCE_IMAGE quando enable_zombie=True e source=None.
ENABLE_ZOMBIE = SOURCE_MODE == "zumbi"
if SOURCE_MODE == "upload":
    source = Path(SOURCE_PATH) if SOURCE_PATH else upload("a foto do rosto de origem", SUPPORTED_IMAGE_EXTENSIONS)
    source = validate_image_path(source)
elif SOURCE_MODE == "zumbi":
    source = None
    assert ZOMBIE_REFERENCE_IMAGE.is_file()
else:
    source = validate_image_path({"vampiro": VAMPIRE_REFERENCE_IMAGE, "ciborgue": CYBORG_REFERENCE_IMAGE}[SOURCE_MODE])

import cv2
cap = cv2.VideoCapture(str(video))
ok, _ = cap.read()
info = dict(w=int(cap.get(3)), h=int(cap.get(4)), fps=cap.get(5), n=int(cap.get(7)))
cap.release()
if not ok:
    raise ValueError("OpenCV não conseguiu ler o primeiro quadro (codec não suportado?). Veja o diagnóstico.")
print(f"Vídeo : {video.name} | {info['w']}x{info['h']} | {info['fps']:.2f} fps | {info['n']} quadros | {video.stat().st_size/1024**2:.1f} MB")
print(f"Rosto : {source.name if source else 'preset zumbi (ZOMBIE_REFERENCE_IMAGE)'}")

## 8️⃣ Otimização de desempenho (configurável)

O que se aplica a **esta** arquitetura:
- **Streaming quadro a quadro**: `VideoFileSource` → pipeline → `VideoFileSink` (FFmpeg via stdin). Vídeos longos não são carregados na RAM.
- **Cache de modelos**: todas as sessões ONNX são criadas uma única vez no `VideoPipeline` e reutilizadas por quadro.
- **Embedding a cada 5 quadros**: o código original só recalcula o ArcFace a cada 5 quadros (`need_embedding`).
- **ROI**: composição e segmentação de cabelo operam só em volta do rosto.
- **Não aplicável**: lotes (o código processa 1 quadro por vez), FP16/AMP (modelos FP32 em ONNX, não PyTorch).

Opcionais abaixo (**alteram o vídeo**, por isso vêm desligados): `MAX_SECONDS` corta o início do vídeo para testes rápidos;
`MAX_HEIGHT` reduz a resolução (menos tempo de CPU em composição/encoding, rosto com menos pixels).

In [ ]:
video_proc = video
if MAX_SECONDS > 0 or MAX_HEIGHT > 0:
    import shutil
    ff = shutil.which("ffmpeg")
    cmd = [ff, "-y", "-i", str(video)]
    if MAX_SECONDS > 0:
        cmd += ["-t", str(MAX_SECONDS)]
    if MAX_HEIGHT > 0 and info["h"] > MAX_HEIGHT:
        cmd += ["-vf", f"scale=-2:{MAX_HEIGHT}"]
    video_proc = INPUT_DIR / f"{video.stem}_prep.mp4"
    cmd += ["-c:v", "libx264", "-crf", "18", "-preset", "veryfast", "-c:a", "copy", str(video_proc)]
    run(cmd)
    print("⚠️ Usando vídeo pré-processado:", video_proc)
else:
    print("Vídeo original será processado sem alterações.")

## 9️⃣ Execução do pipeline original (`VideoPipeline.process`)

In [ ]:
import time, gc, traceback, shutil
from src.pipeline.video_pipeline import VideoPipeline, PipelineProgress
from config.settings import OUTPUT_DIR

def gpu_mem():
    # O ONNX Runtime aloca VRAM fora do PyTorch → torch.cuda.memory_allocated() não a enxerga. Usamos nvidia-smi.
    if not shutil.which("nvidia-smi"):
        return "n/d"
    r = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total,utilization.gpu",
                        "--format=csv,noheader,nounits"], capture_output=True, text=True)
    u, t, g = [x.strip() for x in r.stdout.splitlines()[0].split(",")]
    return f"VRAM {u}/{t} MB | GPU {g}%"

_last = [0.0]
def on_progress(p: PipelineProgress):
    agora = time.time()
    if agora - _last[0] < 2 and p.current_frame != p.total_frames:
        return
    _last[0] = agora
    tot = f"/{p.total_frames}" if p.total_frames else ""
    print(f"[{p.current_frame}{tot}] {p.fps:.1f} fps | {p.elapsed_seconds:.0f}s | rostos={p.faces_detected} "
          f"| alvo={'OK' if p.target_found else '—'} | {gpu_mem()}", flush=True)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
saida = OUTPUT_DIR / f"{video.stem}_swapped.mp4"

t0 = time.perf_counter()
pipeline = VideoPipeline(
    device_preference=DevicePreference(DEVICE),
    on_progress=on_progress,
    enable_enhancer=ENABLE_ENHANCER,
    blend_intensity=BLEND_INTENSITY,
    color_correct=COLOR_CORRECT,
    feather_amount=FEATHER_AMOUNT,
    enable_zombie=ENABLE_ZOMBIE,
    enable_head_swap=ENABLE_HEAD_SWAP,
)
print(f"Modelos carregados em {time.perf_counter()-t0:.1f}s | dispositivo: {pipeline.device_manager.info.name} | {gpu_mem()}")

try:
    t1 = time.perf_counter()
    resultado = pipeline.process(video_path=video_proc, source_image_path=source, output_path=saida)
    TEMPO_TOTAL = time.perf_counter() - t1
    print(f"\n✅ Concluído em {TEMPO_TOTAL:.1f}s → {resultado}")
except Exception:
    print("❌ Falha no processamento:\n" + traceback.format_exc())
    print("Log do FFmpeg (se existir):", sorted(Path("temp").glob("ffmpeg_*.log")))
    raise

## 🔟 Validação

In [ ]:
import json as _json
checks = []
def check(nome, cond, extra=""):
    checks.append((nome, bool(cond)))
    print(f"{'✅' if cond else '❌'} {nome} {extra}")

check("Repositório clonado", (FACELAB / "src").is_dir())
check("Detector em CUDA", "CUDAExecutionProvider" in pipeline.detector.active_providers, pipeline.detector.active_providers)
check("Swapper em CUDA", "CUDAExecutionProvider" in pipeline.swapper.active_providers, pipeline.swapper.active_providers)
if pipeline.parser is not None:
    pp = pipeline.parser._session.get_providers()
    check("BiSeNet em CUDA", "CUDAExecutionProvider" in pp, pp)
if pipeline.enhancer is not None:
    check("GFPGAN em CUDA", "CUDAExecutionProvider" in pipeline.enhancer.active_providers, pipeline.enhancer.active_providers)

resultado = Path(resultado)
check("Arquivo de saída existe", resultado.is_file() and resultado.stat().st_size > 0,
      f"({resultado.stat().st_size/1024**2:.1f} MB)" if resultado.is_file() else "")

probe = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "stream=codec_type,codec_name,width,height,nb_frames",
                        "-show_entries", "format_tags=comment", "-of", "json", str(resultado)],
                       capture_output=True, text=True)
check("Saída legível pelo ffprobe", probe.returncode == 0)
print(probe.stdout)

if not dm.info.is_cuda:
    print("⚠️ Validação executada SEM GPU — os itens 'em CUDA' acima falham por definição.")
print("ℹ️ Os testes pytest do repositório (tests/) dependem de temp/rosto.png, que não é versionado; não são executados aqui.")
falhas = [n for n, ok in checks if not ok]
print("\nResultado:", "TUDO OK" if not falhas else f"FALHAS: {falhas}")

## 1️⃣1️⃣ Resultados e exportação

In [ ]:
from IPython.display import HTML, display
from base64 import b64encode

print(f"Arquivo      : {resultado}")
print(f"Dispositivo  : {pipeline.device_manager.info.name}")
print(f"Tempo total  : {TEMPO_TOTAL:.1f}s")
print("Saídas em output/:", [p.name for p in OUTPUT_DIR.glob('*.mp4')])

# Pré-visualização inline apenas para arquivos pequenos (base64 cresce ~33% e pesa no navegador)
if resultado.stat().st_size < 50 * 1024**2:
    data = b64encode(resultado.read_bytes()).decode()
    display(HTML(f'<video width=640 controls src="data:video/mp4;base64,{data}"></video>'))
else:
    print("Arquivo grande demais para pré-visualizar; faça o download abaixo.")

In [ ]:
BAIXAR_PARA_PC   = False  # @param {type:"boolean"}
COPIAR_PARA_DRIVE = False # @param {type:"boolean"}

if COPIAR_PARA_DRIVE:
    if not USE_DRIVE:
        raise RuntimeError("Ative USE_DRIVE e execute a célula de montagem do Drive primeiro.")
    destino = Path(DRIVE_BASE, "output", resultado.name)
    shutil.copy2(resultado, destino)
    print("Copiado para:", destino)

if BAIXAR_PARA_PC:
    from google.colab import files
    files.download(str(resultado))

# Alternativa manual: painel 📁 à esquerda → FaceLab/output → clique com o botão direito → Fazer download.

In [ ]:
# Liberação de memória ao terminar (as sessões ONNX liberam a VRAM ao serem destruídas)
del pipeline
gc.collect()
dm.release_memory()
print("Memória liberada |", gpu_mem())

## 1️⃣2️⃣ Diagnóstico de erros

| Sintoma | Causa provável | Solução específica |
|---|---|---|
| `ModuleNotFoundError: src` / `config` | Célula 3 não executada nesta sessão (cwd/sys.path) | Execute de novo a célula **Clonar o repositório**. |
| `ModuleNotFoundError: insightface` | Instalação falhou (sem wheel para esta versão do Python) | Veja a saída da célula 4; o erro do pip mostra o motivo. Não reinstale o ambiente inteiro. |
| `CUDAExecutionProvider` ausente / *"CUDA EP indisponível"* | `onnxruntime` CPU instalado junto, ou CUDA/cuDNN incompatíveis | `pip uninstall -y onnxruntime onnxruntime-gpu` e rode a célula 4 de novo. Confirme na célula 2 que o PyTorch usa CUDA 12.x (ORT 1.23.2 exige CUDA 12 + cuDNN 9). |
| `libcudnn*.so: cannot open shared object` | cuDNN não encontrado pelo ORT | Garanta `import torch` antes de `onnxruntime` (célula 5) e `ort.preload_dlls()`. |
| `Failed to allocate memory` / `CUDA out of memory` | VRAM insuficiente (GPU pequena ou outro processo) | Desligue `ENABLE_ENHANCER` (GFPGAN é o maior consumidor) e/ou `ENABLE_HEAD_SWAP`; reinicie o runtime para liberar sessões antigas. |
| `RuntimeError` CUDA com GPU listada | Sessão de GPU revogada / driver | Ambiente de execução → Desconectar e excluir → reconectar com GPU. |
| `FileNotFoundError: Modelo ... não encontrado` | Download incompleto | Apague só o arquivo corrompido em `FaceLab/models/` e rode a célula 6. |
| `HTTP Error 403/404` no download | Hugging Face/GitHub indisponível ou limite | Aguarde e tente novamente; não use espelhos não verificados. |
| `ValidationError: Arquivo ... não encontrado` | Caminho incorreto | Use caminho absoluto (`/content/...`); o cwd é `FaceLab/`. |
| `ffmpeg não encontrado` / erro de escrita | FFmpeg ausente ou codec | `apt-get install -y ffmpeg`; consulte `FaceLab/temp/ffmpeg_*.log`. |
| OpenCV não lê o 1º quadro | Codec raro (ex.: HEVC 10-bit) | Converta: `ffmpeg -i in.mov -c:v libx264 -pix_fmt yuv420p in.mp4`. |
| `Nenhum rosto foi detectado na imagem de referência` | Foto sem rosto frontal nítido | Use foto frontal, bem iluminada; o projeto já tenta limiar 0.2 automaticamente. |
| Sessão reiniciou sozinha | Falta de RAM | Use `MAX_HEIGHT`/`MAX_SECONDS` para testes; feche outras abas do Colab. |
| Sessão desconectada no meio | Limite de tempo/inatividade do Colab | Processe trechos menores (`MAX_SECONDS`) e salve no Drive. |